# MEN 4338 · Week 12 — Indicator kriging: the probability of ore

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W12_indicator_kriging.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

A mine manager rarely asks "what is the grade here?". The real question is **"what is the chance that this
block is ore?"**. Indicator geostatistics answers it directly. Code each sample 1 if it is above the cut-off,
0 otherwise; kriging these 0/1 values gives a **probability**. No assumption on the shape of the histogram is
needed, and extreme grades no longer dominate.

**What you will practise:** indicator coding · declustered proportions · indicator variograms · indicator
kriging · order-relation corrections · probability maps and risk.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('data/bassam_truth_grid.csv', 'bassam_truth_grid.csv'), ('tests/w12/q1.py', 'tests/q1.py'), ('tests/w12/q2.py', 'tests/q2.py'), ('tests/w12/q3.py', 'tests/q3.py'), ('tests/w12/q4.py', 'tests/q4.py'), ('tests/w12/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def exp_variogram(x, y, v, lag, nlags, azimuth=None, atol=22.5):
    """Experimental semivariogram. Lag class k (1..nlags) holds the pairs with |d - k*lag| < lag/2.
    With an azimuth (degrees clockwise from north), only pairs within +/- atol of that direction are kept.
    Returns mean distance, semivariance and number of pairs of each class (NaN where a class is empty)."""
    x, y, v = (np.asarray(a, float) for a in (x, y, v))
    i, j = np.triu_indices(len(v), 1)
    dx, dy = x[j] - x[i], y[j] - y[i]
    d = np.hypot(dx, dy)
    keep = np.ones(len(d), bool)
    if azimuth is not None:
        az = np.degrees(np.arctan2(dx, dy)) % 180
        diff = np.abs(az - azimuth % 180)
        keep = np.minimum(diff, 180 - diff) <= atol
    sq = 0.5 * (v[j] - v[i]) ** 2
    hm, g, n = np.full(nlags, np.nan), np.full(nlags, np.nan), np.zeros(nlags, int)
    for k in range(1, nlags + 1):
        m = keep & (np.abs(d - k * lag) < lag / 2)
        n[k - 1] = m.sum()
        if n[k - 1]:
            hm[k - 1], g[k - 1] = d[m].mean(), sq[m].mean()
    return hm, g, n

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

def cell_weights(x, y, cell_size):
    """Cell declustering weights (sum to 1); cells start at the origin."""
    key = np.floor(np.asarray(x) / cell_size).astype(int) * 100000 + np.floor(np.asarray(y) / cell_size).astype(int)
    _, inv, cnt = np.unique(key, return_inverse=True, return_counts=True)
    w = 1.0 / cnt[inv]
    return w / w.sum()

def ok_point(x0, y0, x, y, v, gamma, nmax=16):
    """Ordinary kriging at (x0, y0) with the nmax nearest data.
    gamma(dx, dy) is the variogram model, a function of the separation vector.
    Returns (estimate, kriging variance)."""
    d = np.hypot(x - x0, y - y0)
    idx = np.argsort(d)[:nmax]
    xs, ys, vs = x[idx], y[idx], v[idx]
    n = len(idx)
    A = np.ones((n + 1, n + 1)); A[n, n] = 0
    A[:n, :n] = gamma(xs[:, None] - xs[None, :], ys[:, None] - ys[None, :])
    b = np.ones(n + 1); b[:n] = gamma(xs - x0, ys - y0)
    sol = np.linalg.solve(A, b)
    w, mu = sol[:n], sol[n]
    return float(w @ vs), float(w @ b[:n] + mu)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

samples = pd.read_csv("bassam_samples.csv")
x, y, au = samples["x"].to_numpy(), samples["y"].to_numpy(), samples["au_gpt"].to_numpy()
is_grid = (samples["campaign"] == "grid").to_numpy()
T = pd.read_csv("bassam_truth_grid.csv")["au_gpt"].to_numpy().reshape(100, 120)

## 1. Indicators

$$i(\mathbf{x}; z_c) = \begin{cases} 1 & z(\mathbf{x}) > z_c \\ 0 & \text{otherwise} \end{cases}$$

The (declustered) mean of an indicator is the proportion of the deposit above the cut-off. Its variance is
$p(1-p)$, which is also the sill of its variogram.

## 2. Indicator kriging (IK)

Ordinary kriging of the indicators with the **indicator variogram** gives, at each location, an estimate of
$P\{Z(\mathbf{x}) > z_c \mid \text{data}\}$. Because kriging weights can be negative, the result can fall
slightly outside $[0, 1]$: clip it (an *order-relation correction*). With several cut-offs, the probabilities
must also decrease as the cut-off increases.

The indicator variogram at 1 g/t on the Bassam grid holes is modelled by `gamma_ind`: nugget 0.08 +
spherical 0.17, range 280 m along N045 and 130 m along N135.

In [ ]:
def gamma_ind(dx, dy):
    return spherical(aniso_h(dx, dy, 45, 280, 130), 0.08, 0.17, 280)

---
## Exercises

### Q1. Indicators and proportions

Store the indicator of `au > 1.0` as floats in `ind_1`, its plain mean in `p_naive` and its declustered mean
(50 m cells) in `p_declus`. Then build the dictionary `props` with the declustered proportion above 0.5, 1.0
and 1.5 g/t.

In [ ]:
ind_1 = ...
p_naive = ...
p_declus = ...
props = ...
p_naive, p_declus, props

In [ ]:
grader.check("q1")

### Q2. Indicator variogram

On the grid holes, compute the omnidirectional indicator variogram (lag 50 m, 10 lags) in `g_ind` and the
indicator variance $p(1-p)$ in `ind_sill` (with $p$ = mean of `ind_1` on the grid holes). Plot them with the
model `gamma_ind` along N045 and N135.

In [ ]:
g_ind = ...
ind_sill = ...

In [ ]:
grader.check("q2")

### Q3. Probability at three points

Store in `ik_targets` the probability of exceeding 1 g/t at (312, 457), (641, 213) and (906, 782): `ok_point`
on `ind_1` with `gamma_ind` and 16 neighbours, clipped to $[0, 1]$.

In [ ]:
ik_targets = ...
ik_targets

In [ ]:
grader.check("q3")

### Q4. Probability map

Krig the probability on the week 9 nodes (`np.meshgrid(np.arange(15, 1200, 20), np.arange(15, 1000, 20))`)
into `prob_map` (shape 50 × 60, clipped), draw it, and store in `ha_above_05` the area in hectares where the
probability is above 0.5.

In [ ]:
gx, gy = np.meshgrid(np.arange(15, 1200, 20), np.arange(15, 1000, 20))
prob_map = ...
ha_above_05 = ...
ha_above_05

In [ ]:
grader.check("q4")

### Q5. How good are the probabilities?

With `truth_nodes = T[1::2, 1::2]`: store the true share of nodes above 1 g/t in `true_frac`, the expected
share from the probabilities (`prob_map.mean()`) in `expected_frac`, and the share of nodes misclassified by
the rule "ore if probability > 0.5" in `misclass`.

In [ ]:
truth_nodes = T[1::2, 1::2]
true_frac = ...
expected_frac = ...
misclass = ...
true_frac, expected_frac, misclass

In [ ]:
grader.check("q5")

### Going further (not checked)

* Check calibration: group the nodes by probability (0–0.1, 0.1–0.2, …) and compute the true share of ore in
  each group. A well-calibrated map gives about 0.15 in the 0.1–0.2 group, and so on.
* Krig the indicators at 0.5 and 1.5 g/t too (use the same model for simplicity). Are the three probabilities
  always in decreasing order? Fix the nodes where they are not.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()